### Loading Libraries

In [69]:
import os
from dotenv import load_dotenv
import praw
from openai import OpenAI
import pandas as pd
import time
from pinecone import Pinecone
import json

### Connect APIs

In [52]:
### Setup
load_dotenv('../.env.local')
REDDIT_CLIENT_ID = os.environ.get('REDDIT_CLIENT_ID')
REDDIT_SECRET_ID = os.environ.get('REDDIT_SECRET_ID')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')


In [84]:
### Reddit API
reddit = praw.Reddit(
    client_id = REDDIT_CLIENT_ID,
    client_secret = REDDIT_SECRET_ID,
    user_agent = "researcher"
)
# Session options: controversial, gilded, hot, new, rising, top
print(reddit.read_only)

### OpenAI API
client = OpenAI(api_key=OPENAI_API_KEY)

### Pinecone API
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index('flight-reviews')

True


In [91]:
for submission in reddit.subreddit("travel").hot(limit=2):
    print(submission.title)
    print(submission.selftext)
    print('---')

Reminder: any use of ChatGPT or AI tools will result in a ban
Mods are seeing a noticeable increase in users using ChatGPT and similar tools not only to create posts but also to post *entire* responses in comments, disguised as genuine personal advice.

The sub is one of the biggest on Reddit and as a community it's so important - particularly for a topic like travel which is rooted in authentic human experiences - that all responses come in the form of genuine opinions and guidance. There's absolutely no point in us all being on here otherwise.

Mods have tools to identify these sort of posts, but it's worth reiterating moving into 2025 and with increased AI available in our day-to-day lives that any usage of this sort to make your posts or comments will result in an instant ban. The rules are stated very clearly in the sidebar and are not new.

None of us joined this community to read regurgitated information from a machine learning model like ChatGPT. AI tools can have their place f

### Scrape Reddit

In [76]:
### get comments and posts
target_subreddits = reddit.subreddit("delta+united+AmericanAirlines+travel+awardtravel")
search_query = ''' 
                ("Business Class" OR "First Class" OR "Status" OR "Priority") AND 
                ("App" OR "Website" OR "Boarding" OR "Wifi" OR "Check-in" OR "Glitch") 
                '''

title, comment, score = [], [], []
for submission in target_subreddits.search(
    search_query,
    sort="relevance",
    time_filter="year",
    limit=10):
    # print(submission.title)
    # print("comments:")
    submission.comments.replace_more(limit=1)
    for i in range(len(submission.comments.list())):
        # print(submission.comments.list()[i].body)
        # print(submission.score)
        # print("--")
        title.append(submission.title)
        comment.append(submission.comments.list()[i].body)
        score.append(submission.score)
        print(f'Unique Submissions {len(set(title))}, total comments {len(comment):,}', end='\r', flush=True)
    print(f'Unique Submissions {len(set(title))}, total comments {len(comment):,}', end='\r', flush=True)
        # time.sleep()

In [77]:
reddit_df = pd.DataFrame({
    'Title': title,
    'Comment': comment,
    'Score': score
})
display(reddit_df)

,Title,Comment,Score
0,Passenger tried to pull rank on me,LOL I always fly bulkhead. If wishes were fish...,12528
1,Passenger tried to pull rank on me,"This reads like chatGPT, 100%",12528
2,Passenger tried to pull rank on me,[deleted],12528
3,Passenger tried to pull rank on me,And then everyone clapped and carried me out o...,12528
4,Passenger tried to pull rank on me,While I always appreciate a good bit of fictio...,12528
...,...,...,...
3533,Glad to see Delta empowers their employees to ...,Did you just call me a queef? How dare you.,2516
3534,Glad to see Delta empowers their employees to ...,">If they made every person size their carryon,...",2516
3535,Glad to see Delta empowers their employees to ...,False. Yes it does,2516
3536,Glad to see Delta empowers their employees to ...,"This solves the problem of too big bags, but n...",2516


### Vectorize Comments

In [79]:
# Filter out empty/removed comments
clean_comments = [c for c in comment if c not in ['[removed]', '[deleted]'] and len(c.strip()) > 0]
print(f"Total comments to embed: {len(clean_comments):,}")

Total comments to embed: 3,491


##### Batch Embeddings

In [80]:
# Set batch and comment char length
batch_size = 200
max_chars = 200000

all_vectors = []
total_batches = (len(clean_comments) + batch_size - 1) // batch_size
print(f'Processing {total_batches} batches of up to {batch_size} comments')

for batch_num in range(0, len(clean_comments), batch_size):
    batch = clean_comments[batch_num:batch_num + batch_size]
    batch_char = sum(len(c) for c in batch)

    print(f'Batch {(batch_num // batch_size) + 1}/{total_batches}: Embedding {len(batch)} comments ({batch_char:,} chars...', end=" ")

    embeddings_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    vectors = [
        (f'id_{batch_num}_{j}', embeddings_response.data[j].embedding, {'text': batch[j]})
        for j in range(len(batch))
    ]

    all_vectors.extend(vectors)
    print(f'Done ({len(all_vectors):,} vectors total)')

Processing 18 batches of up to 200 comments
Batch 1/18: Embedding 200 comments (21,159 chars... Done (200 vectors total)
Batch 2/18: Embedding 200 comments (21,782 chars... Done (400 vectors total)
Batch 3/18: Embedding 200 comments (29,835 chars... Done (600 vectors total)
Batch 4/18: Embedding 200 comments (41,979 chars... Done (800 vectors total)
Batch 5/18: Embedding 200 comments (36,466 chars... Done (1,000 vectors total)
Batch 6/18: Embedding 200 comments (34,834 chars... Done (1,200 vectors total)
Batch 7/18: Embedding 200 comments (38,077 chars... Done (1,400 vectors total)
Batch 8/18: Embedding 200 comments (30,190 chars... Done (1,600 vectors total)
Batch 9/18: Embedding 200 comments (25,462 chars... Done (1,800 vectors total)
Batch 10/18: Embedding 200 comments (39,724 chars... Done (2,000 vectors total)
Batch 11/18: Embedding 200 comments (32,932 chars... Done (2,200 vectors total)
Batch 12/18: Embedding 200 comments (47,886 chars... Done (2,400 vectors total)
Batch 13/18: 

##### Upsert to Pinecone

In [81]:
upsert_batch_size = 100
for i in range(0, len(all_vectors), upsert_batch_size):
    batch = all_vectors[i:i+upsert_batch_size]
    index.upsert(vectors=batch)
    print(f'Uploading {i + len(batch)}/{len(all_vectors)} vectors')
print(f'Successfully uploaded {len(all_vectors)} vectors to Pinecone')

Uploading 100/3491 vectors
Uploading 200/3491 vectors
Uploading 300/3491 vectors
Uploading 400/3491 vectors
Uploading 500/3491 vectors
Uploading 600/3491 vectors
Uploading 700/3491 vectors
Uploading 800/3491 vectors
Uploading 900/3491 vectors
Uploading 1000/3491 vectors
Uploading 1100/3491 vectors
Uploading 1200/3491 vectors
Uploading 1300/3491 vectors
Uploading 1400/3491 vectors
Uploading 1500/3491 vectors
Uploading 1600/3491 vectors
Uploading 1700/3491 vectors
Uploading 1800/3491 vectors
Uploading 1900/3491 vectors
Uploading 2000/3491 vectors
Uploading 2100/3491 vectors
Uploading 2200/3491 vectors
Uploading 2300/3491 vectors
Uploading 2400/3491 vectors
Uploading 2500/3491 vectors
Uploading 2600/3491 vectors
Uploading 2700/3491 vectors
Uploading 2800/3491 vectors
Uploading 2900/3491 vectors
Uploading 3000/3491 vectors
Uploading 3100/3491 vectors
Uploading 3200/3491 vectors
Uploading 3300/3491 vectors
Uploading 3400/3491 vectors
Uploading 3491/3491 vectors
Successfully uploaded 3491 ve

### Query Relevant Comments

In [82]:
query = "Business class flyers care about their online boarding experience."
query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
    ).data[0].embedding

results = index.query(vector=query_embedding, top_k=3, include_metadata=True)

for match in results['matches']:
    print(f'Score: {match['score']:.3f}')
    print(f'Comment {match['metadata']['text']}\n')

Score: 0.549
Comment Just board earlier. Pay for priority boarding if getting into your assigned seat matters to you.

Score: 0.529
Comment Did they offer you their business class seats?

Score: 0.525
Comment They need to start making people sign an agreement that you will sit in the seat you were assigned and/or paid for, put it multiple places on the website, app, boarding pass, signs in the airport, and multiple announcements during boarding. It’s not a public bus. It’s a price dependent reservation.



In [83]:
results_dict = {}
for match in results['matches']:
    results_dict[match['score']] = match['metadata']['text']

with open('../datasets/reddit/top_comments.json', 'w') as file:
    json.dump(results_dict, file, indent=4)